In [ ]:
import torch
from torch import optim
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.preprocessing import OneHotEncoder
!mkdir /content/models

In [ ]:
# 모델 최적화
'''
1. 내부 공변량 문제
내부 공변량: 모델 학습 시 입력값으로 들어오는 배치마다 입력 분포(평균, 편차)가 변화하는 현상
입력값에 정규화가 이루어져 있지 않은 경우 가중치가 제대로 갱신되지 않거나 학습이 불안정해지고 느려질 수 있다.
'''
# 모델 최적화
'''
1. 내부 공변량 문제
내부 공변량: 모델 학습 시 입력값으로 들어오는 배치마다 입력 분포(평균, 편차)가 변화하는 현상
입력값에 정규화가 이루어져 있지 않은 경우 가중치가 제대로 갱신되지 않거나 학습이 불안정해지고 느려질 수 있다.
'''

In [ ]:
# 코드 예시
x = torch.FloatTensor(
    [
        [1.0, 2.0, 9.0],
        [-5.0, 0.0, 5.0],
        [10.0, 20.0, 30.0],
        [12.0, 32.0, 132.0]
    ]
)

In [ ]:
BatchNorm = torch.nn.BatchNorm1d(3)
print(BatchNorm(x)) # 세로 축 기준으로 정규화

LayerNorm1 = torch.nn.LayerNorm(normalized_shape = 3)
print(LayerNorm1(x)) # 가로 축 기준으로 정규화
LayerNorm2 = torch.nn.LayerNorm(normalized_shape = [4, 3]) # 전체 기준으로 정규화
print(LayerNorm2(x))

tensor([[-0.5092, -0.8699, -0.6772],
        [-1.3820, -1.0212, -0.7545],
        [ 0.8001,  0.4917, -0.2709],
        [ 1.0911,  1.3995,  1.7026]], grad_fn=<NativeBatchNormBackward0>)
tensor([[-0.8429, -0.5620,  1.4049],
        [-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247],
        [-0.8890, -0.5080,  1.3970]], grad_fn=<NativeLayerNormBackward0>)
tensor([[-0.5562, -0.5279, -0.3300],
        [-0.7259, -0.5845, -0.4431],
        [-0.3017, -0.0189,  0.2640],
        [-0.2451,  0.3205,  3.1487]], grad_fn=<NativeLayerNormBackward0>)


In [ ]:
# 가중치 초기화
# 1. 상수 초기화 : 대칭파괴(Breaking Symmetry); 모든 노드가 동일한 출력을 생성하는 현상 문제 발생
# 2. 무작위 초기화
# 3. Xavier, He 초기화

# 파이토치 자체도 기본 가중치 초기화를 지원하지만 ReLu 계열 활성화 함수를 사용할 때에는 xavier나 he 방식으로 해주는 게 좋다.

In [ ]:
# 실습1
class MyModel1(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer = nn.Sequential(
        nn.Linear(1,2),
        nn.Sigmoid()
    )
    self.fc = nn.Linear(2,1)
    self._init_weights()

  def forward(self, x):
    output = self.layer(x)
    output = self.fc(output)
    return output

  def _init_weights(self):
    nn.init.xavier_uniform_(self.layer[0].weight) # 가중치와 편향이 있는 Linear 게층만 초기화해야 한다.
    self.layer[0].bias.data.fill_(0.01) # 편향 값

    nn.init.xavier_uniform_(self.fc.weight)
    self.fc.bias.data.fill_(0.01) # 편향 값

model = MyModel1()
print(model)

MyModel1(
  (layer): Sequential(
    (0): Linear(in_features=1, out_features=2, bias=True)
    (1): Sigmoid()
  )
  (fc): Linear(in_features=2, out_features=1, bias=True)
)


In [ ]:
# 실습2
# apply 함수 사용 (모델의 모든 하위 모듈(submodule)에 사용자 정의 함수를 재귀적으로 적용할 때 사용)
class MyModel1(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer = nn.Sequential(
        nn.Linear(1,2),
        nn.Sigmoid()
    )
    self.fc = nn.Linear(2,1)
    self.apply(self._init_weights)

  def forward(self, x):
    output = self.layer(x)
    output = self.fc(output)
    return output

  def _init_weights(self, module):
    if isinstance(module, nn.Linear):
      nn.init.xavier_uniform_(self.layer[0].weight) # 가중치와 편향이 있는 Linear 게층만 초기화해야 한다.
      nn.init.constant_(module.bias, 0.01)
      print('가중치 초기화 적용 완료')

model = MyModel1()
print(model)

가중치 초기화 적용 완료
가중치 초기화 적용 완료
MyModel1(
  (layer): Sequential(
    (0): Linear(in_features=1, out_features=2, bias=True)
    (1): Sigmoid()
  )
  (fc): Linear(in_features=2, out_features=1, bias=True)
)


In [ ]:
'''
for x, y in dataloader:
  x = x.to(device)
  y = y.to(device)
  output = model(x)

  _lambda = 0.5
  l2_loss = sum(p.pow(2).sum() for p in model.parameters())

  loss = criterion(output, y) + _lambda * l2_loss

optimizer = torch.optim.SGD(model.parameters(), lr = 0.01, weight_decay = 0.01)

class MyModel1(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer1 = nn.Linear(10, 10)
    self.dropout = nn.Dropout(p = 0.5) # layer1에서 50프로의 노드를 0으로 만들어버림
    self.layer 2 = nn.Linear(10, 10)

  def forward(self, x):
    x = self.layer1(x)
    x = self.dropout(x)
    x = self.layer2(x)

  for x, y in train_dataloader:
    x = x.to(device)
    y = y.to(device)
    output = model(x)

    loss = criterion(output, y)
    optimizer.zero_grad()
    loss.backward()

    torch.nn.utils.clip_grad_norm(model.parameters(), 0.1)

    optimizer.step()

    cost += loss

'''